# Library

In [1]:
# import packages
import numpy as np
np.random.seed(4)
import matplotlib.pyplot as plt
import seaborn as sns

from itertools import groupby
import pandas as pd
import datetime as dt

import os
import re
import string
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from transformers import pipeline
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from nltk.tokenize import word_tokenize, sent_tokenize

import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm as tq
tq.pandas() #thanks to https://stackoverflow.com/questions/18603270/progress-indicator-during-pandas-operations


# Abstractness

In [2]:
from wordtangible import word_concreteness, avg_text_concreteness, concrete_abstract_ratio

def concreteAvg(doc):
    result = avg_text_concreteness(doc)
    if result == 0.0:
        return np.nan
    else: 
        return result

def concreteAvgExtra(doc):
    result = avg_text_concreteness(doc, only_rated_words = False)
    if result == 0.0:
        return np.nan
    else: 
        return result

def concreteRatio(doc):
    result = concrete_abstract_ratio(doc)
    if result == np.inf:
        return 5.0
    else:
        return result


# Get concreteness rating for a single word
print(word_concreteness("might"))  # Output: 5.0 (highly concrete)
print(word_concreteness("will"))  # Output: 5.0 (highly concrete)



2.32
2.64


In [3]:
exList = ["The Committee will set the rate at 0.25 percent until inflation reaches 2 percent and unemployment falls below 5 percent. We expect these conditions to be met by mid-2026.",
          "The Committee anticipates that maintaining the current policy rate may be appropriate as long as inflation trends toward 2 percent and labor market indicators improve. These conditions could potentially be met by mid-2026, although considerable uncertainty remains."]
# Calculate average concreteness of a text
for text in exList:
    print(6-concreteAvg(text))
    print(1/concreteRatio(text))


3.173
1.0
3.4085
3.0


# Readability, Informativeness, Disunity

In [6]:
try:
    import textdescriptives
except:
    !pip install "textdescriptives[tutorials]"
import textdescriptives as td
import spacy

In [7]:
exList = ["The Committee will set the rate at 0.25 percent until inflation reaches 2 percent and unemployment falls below 5 percent. We expect these conditions to be met by mid-2026.",
          "The Committee anticipates that maintaining the current policy rate may be appropriate as long as inflation trends toward 2 percent and labor market indicators improve. These conditions could potentially be met by mid-2026, although considerable uncertainty remains.",
          "The Committee will maintain the policy rate at its current level until inflation reaches 2 percent and unemployment falls below 5 percent. These thresholds reflect our dual mandate and are consistent with our medium-term objectives.",
          "The Committee will maintain the policy rate at its current level until inflation reaches 2 percent and unemployment falls below 5 percent. Financial markets have recently shown increased volatility, and global supply chains remain under pressure."]

metrics = td.extract_metrics(
    text=exList,
    spacy_model="en_core_web_sm",
    metrics=["descriptive_stats","readability", "dependency_distance","coherence","information_theory","quality"],
)

In [15]:
print(metrics.columns.tolist())

['text', 'entropy', 'perplexity', 'per_word_perplexity', 'flesch_reading_ease', 'flesch_kincaid_grade', 'smog', 'gunning_fog', 'automated_readability_index', 'coleman_liau_index', 'lix', 'rix', 'dependency_distance_mean', 'dependency_distance_std', 'prop_adjacent_dependency_relation_mean', 'prop_adjacent_dependency_relation_std', 'first_order_coherence', 'second_order_coherence', 'passed_quality_check', 'n_stop_words', 'alpha_ratio', 'mean_word_length', 'doc_length', 'symbol_to_word_ratio_#', 'proportion_ellipsis', 'proportion_bullet_points', 'contains_lorem ipsum', 'duplicate_line_chr_fraction', 'duplicate_paragraph_chr_fraction', 'duplicate_ngram_chr_fraction_5', 'duplicate_ngram_chr_fraction_6', 'duplicate_ngram_chr_fraction_7', 'duplicate_ngram_chr_fraction_8', 'duplicate_ngram_chr_fraction_9', 'duplicate_ngram_chr_fraction_10', 'top_ngram_chr_fraction_2', 'top_ngram_chr_fraction_3', 'top_ngram_chr_fraction_4', 'oov_ratio', 'token_length_mean', 'token_length_median', 'token_length_

In [19]:
metrics[['text','entropy','flesch_kincaid_grade','coleman_liau_index','lix','first_order_coherence']]

,text,entropy,flesch_kincaid_grade,coleman_liau_index,lix,first_order_coherence
0,The Committee will set the rate at 0.25 percen...,0.648819,8.375345,10.747586,45.534483,0.327459
1,The Committee anticipates that maintaining the...,0.772680,13.630405,18.515676,61.743243,0.430545
2,The Committee will maintain the policy rate at...,0.680321,10.113333,14.405556,54.111111,0.493268
3,The Committee will maintain the policy rate at...,0.734654,11.424444,16.528889,56.888889,0.523800
